In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType, LongType,
    DecimalType, DateType, TimestampType, BooleanType,
)

In [0]:
dbutils.widgets.text("environment", "dev")
env = dbutils.widgets.get("environment")
print(env)

In [0]:
sources = ['branch', 'customer_history', 'transaction', 'account', 'product', 'transaction_line',
           'account_customer_history', 'batch_manifest', 'daily_account_balance']

In [0]:
SOURCE_SCHEMAS = {
    "branch.csv": StructType([
        StructField("branch_id", StringType(), False),
        StructField("branch_name", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("region", StringType(), True),
        StructField("source_modified_at", TimestampType(), True),
    ]),
    "product.csv": StructType([
        StructField("product_id", StringType(), False),
        StructField("product_name", StringType(), True),
        StructField("product_category", StringType(), True),
        StructField("annual_fee", DecimalType(10, 2), True),
    ]),
    "customer_history.csv": StructType([
        StructField("customer_id", StringType(), False),
        StructField("full_name", StringType(), True),
        StructField("address_line_1", StringType(), True),
        StructField("city", StringType(), True),
        StructField("state", StringType(), True),
        StructField("postal_code", StringType(), True),
        StructField("customer_segment", StringType(), True),
        StructField("kyc_status", StringType(), True),
        StructField("effective_from_date", DateType(), False),
        StructField("source_modified_at", TimestampType(), True),
        StructField("source_system", StringType(), True),
    ]),
    "account.csv": StructType([
        StructField("account_id", StringType(), False),
        StructField("primary_customer_id", StringType(), True),
        StructField("branch_id", StringType(), True),
        StructField("account_type", StringType(), True),
        StructField("account_status", StringType(), True),
        StructField("open_date", DateType(), True),
        StructField("close_date", DateType(), True),
        StructField("currency_code", StringType(), True),
        StructField("source_modified_at", TimestampType(), True),
    ]),
    "account_customer_history.csv": StructType([
        StructField("account_id", StringType(), False),
        StructField("customer_id", StringType(), False),
        StructField("ownership_role", StringType(), True),
        StructField("ownership_percent", IntegerType(), True),
        StructField("effective_from_date", DateType(), False),
        StructField("source_modified_at", TimestampType(), True),
        StructField("source_system", StringType(), True),
    ]),
    "transaction.csv": StructType([
        StructField("transaction_id", StringType(), False),
        StructField("account_id", StringType(), True),
        StructField("transaction_timestamp", TimestampType(), True),
        StructField("transaction_date", DateType(), True),
        StructField("transaction_type", StringType(), True),
        StructField("transaction_amount", DecimalType(12, 2), True),
        StructField("currency_code", StringType(), True),
        StructField("merchant_name", StringType(), True),
        StructField("merchant_category", StringType(), True),
        StructField("transaction_status", StringType(), True),
        StructField("source_modified_at", TimestampType(), True),
        StructField("arrival_timestamp", TimestampType(), True),
        StructField("is_late_arrival", BooleanType(), True),
        StructField("source_system", StringType(), True),
    ]),
    "transaction_line.csv": StructType([
        StructField("transaction_id", StringType(), False),
        StructField("line_number", IntegerType(), False),
        StructField("product_id", StringType(), True),
        StructField("quantity", IntegerType(), True),
        StructField("line_amount", DecimalType(12, 2), True),
        StructField("source_modified_at", TimestampType(), True),
    ]),
    "daily_account_balance.csv": StructType([
        StructField("balance_date", DateType(), False),
        StructField("account_id", StringType(), False),
        StructField("currency_code", StringType(), True),
        StructField("available_balance_amount", DecimalType(12, 2), True),
        StructField("ledger_balance_amount", DecimalType(12, 2), True),
        StructField("overdraft_limit_amount", DecimalType(12, 2), True),
        StructField("source_modified_at", TimestampType(), True),
        StructField("source_system", StringType(), True),
    ]),
    "batch_manifest.csv": StructType([
        StructField("batch_id", StringType(), False),
        StructField("processing_date", DateType(), True),
        StructField("load_type", StringType(), True),
        StructField("generated_at_utc", TimestampType(), True),
        StructField("transaction_record_count", LongType(), True),
        StructField("late_arrival_record_count", LongType(), True),
    ]),
}

CSV_READ_OPTIONS = {
    "header": "true",
    "nullValue": "",
    "dateFormat": "yyyy-MM-dd",
    "timestampFormat": "yyyy-MM-dd'T'HH:mm:ssXXX",
    "mode": "FAILFAST",  # Malformed rows abort the read instead of silently nulling them.
}

In [0]:
for source in sources:
    print(source)
    df = spark.read.option("header", "true").schema(SOURCE_SCHEMAS[f"{source}.csv"])\
        .csv(f'/Volumes/{env}_banking/data_source/csv/{source}.csv')
    df.write.format("delta").mode("append").saveAsTable(f"{env}_banking.data_source.{source}")
